# Notebook 03 — Event-Conditioned Reliability Analysis
**Thesis:** Chapter 7 | **Input:** `data/2_events.csv` | **Output:** `data/3_results.csv`

---

## Purpose — answering the three research questions

| RQ | Question | Analysis |
|----|---------|---------|
| RQ1 | What does reconstruction reveal about campaign-wide reliability? | §7.1 — global statistics, rolling PDR, burst distribution |
| RQ2 | How does reliability vary across environmental and operational events? | §7.3–7.6 — event-conditioned PDR for all 18 events |
| RQ3 | Which conditions are most strongly associated with loss risk? | §7.7 — logistic regression, odds ratios |

## Filtered dataset
All event-conditioned analysis uses the **radio-layer filtered dataset**:
```python
df_link = df[~df['is_outage'] & ~df['is_sf_artifact']]
```
This ensures that observed PDR variation is attributable to radio-channel conditions,  
not infrastructure downtime or SF rotation artifacts.

## 0 · Imports & Load

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / '2_events.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values(['device_id', 'time']).reset_index(drop=True)

# filtered dataset — radio-layer losses only (excludes outages and SF artifacts)
df_link = df[~df['is_outage'] & ~df['is_sf_artifact']].copy().reset_index(drop=True)

print(f'Full dataset : {len(df):,} rows')
print(f'Filtered (df_link): {len(df_link):,} rows')
print(f'Excluded     : {len(df)-len(df_link):,} rows (outages + SF artifacts)')
print(f'Devices      : {sorted(df_link["device_id"].unique())}')

Full dataset : 1,217,313 rows
Filtered (df_link): 1,156,407 rows
Excluded     : 60,906 rows (outages + SF artifacts)
Devices      : ['ED0', 'ED1', 'ED2', 'ED3', 'ED4', 'ED5']


## 1 · Global Reconstruction Results  *(§7.1 — RQ1)*

Campaign-wide summary: how much was lost, and what kind of losses were they?

Three PDR values tell the decomposition story:
- **PDR_system** — application layer perspective (everything included)
- **PDR_raw** — radio layer, all losses
- **PDR_link** — radio layer, genuine environmental losses only

In [2]:
# global PDR decomposition
rx    = len(df)
lost_raw   = int(df['mac_to_radio_loss'].sum())
lost_link  = int(df_link['mac_to_radio_loss'].sum())
lost_system= int(df['total_loss'].sum())

pdr_system = rx / (rx + lost_system) * 100
pdr_raw    = rx / (rx + lost_raw)    * 100
pdr_link   = len(df_link) / (len(df_link) + lost_link) * 100

print('=== Campaign-Wide PDR Decomposition ===')
print(f'PDR_system (incl. app-MAC drops)     : {pdr_system:.2f}%')
print(f'PDR_raw    (radio, all losses)        : {pdr_raw:.2f}%')
print(f'PDR_link   (radio, env. losses only)  : {pdr_link:.2f}')
print(f'Gap PDR_system → PDR_link               : {pdr_link-pdr_system:.2f} %')
print(f'\n=== Loss Zone Breakdown ===')
zone_counts = df['loss_zone'].value_counts()
for zone, cnt in zone_counts.items():
    print(f'  {zone:<15}: {cnt:>10,}  ({cnt/len(df)*100:.2f}%)')

=== Campaign-Wide PDR Decomposition ===
PDR_system (incl. app-MAC drops)     : 59.26%
PDR_raw    (radio, all losses)        : 65.83%
PDR_link   (radio, env. losses only)  : 96.36
Gap PDR_system → PDR_link               : 37.10 %

=== Loss Zone Breakdown ===
  no_loss        :  1,135,043  (93.24%)
  sf_artifact    :     60,803  (4.99%)
  radio_loss     :     20,866  (1.71%)
  ambiguous      :        498  (0.04%)
  outage         :        103  (0.01%)


## 2 · Per-Device PDR Summary  *(Table 7.1)*

PDR_link per device — the metric used for all event-conditioned analysis.  
The near-uniform PDR across vastly different link budgets is the first evidence  
for the collision-dominated loss hypothesis.

In [3]:
print(f"{'Device':<8} {'Dist(m)':^8} {'CW':^4} {'WW':^4} {'Received':^10} "
      f"{'Link Lost':^11} {'PDR_link':^10}")
print('-' * 65)

pdrs = {}
for dev, g in df_link.groupby('device_id'):
    rx   = len(g)
    lost = int(g['mac_to_radio_loss'].sum())
    pdr  = rx / (rx + lost) * 100
    pdrs[dev] = pdr
    dist = int(g['distance'].iloc[0])
    cw   = int(g['c_walls'].iloc[0])
    ww   = int(g['w_walls'].iloc[0])
    print(f"{dev:<8} {dist:^8} {cw:^4} {ww:^4} {rx:^10,} {lost:^11,} {pdr:^10.2f}%")

print('-' * 65)
pdr_range = max(pdrs.values()) - min(pdrs.values())
print(f"PDR range across devices: {pdr_range:.2f} pp")
print(f"\nKey observation: {pdr_range:.2f} pp spread across devices with very different")
print(f"link budgets (10m direct LoS to 40m through 4 walls) — consistent with")
print(f"collision-dominated losses rather than signal quality degradation.")

Device   Dist(m)   CW   WW   Received   Link Lost   PDR_link 
-----------------------------------------------------------------
ED0         10     0    0    192,030      7,185      96.39   %
ED1         8      1    0    191,072      7,547      96.20   %
ED2         23     0    2    193,876      6,479      96.77   %
ED3         18     1    2    190,340      7,652      96.14   %
ED4         37     0    5    189,995      7,996      95.96   %
ED5         40     2    2    199,094      6,848      96.67   %
-----------------------------------------------------------------
PDR range across devices: 0.80 pp

Key observation: 0.80 pp spread across devices with very different
link budgets (10m direct LoS to 40m through 4 walls) — consistent with
collision-dominated losses rather than signal quality degradation.


## 3 · Statistical Analysis Core Function  *(§5.5.1)*

All event-conditioned comparisons use the same statistical framework:

1. **PDR computation** — for each event condition, compute mean PDR
2. **Mann-Whitney U test** — nonparametric test on `total_tx` distributions  
   (chosen because loss values are discrete and non-normal)
3. **Effect size** — rank-biserial correlation r = 1 − 2U/(n₁n₂)  
   ranges from -1 (event group has lower PDR) to +1 (event group has higher PDR)
4. **Significance** — α = 0.05 two-sided

This function is called for every binary event comparison in sections 4–8.

In [4]:
def pdr_compare(df_in, event_col, group1_label, group0_label=None):
    """
    Compare PDR between two event groups.
    Returns dict with PDR values, Mann-Whitney U test, and effect size.
    """
    g1 = df_in[df_in[event_col] == group1_label]
    g0 = df_in[df_in[event_col] != group1_label] if group0_label is None \
         else df_in[df_in[event_col] == group0_label]

    def pdr(g):
        rx = len(g)
        lost = int(g['mac_to_radio_loss'].sum())
        return rx / (rx + lost) * 100 if (rx + lost) > 0 else 0

    pdr1, pdr0 = pdr(g1), pdr(g0)
    diff = pdr1 - pdr0

    # Mann-Whitney U on total_tx — tests whether loss distributions differ
    stat, p = stats.mannwhitneyu(
        g1['total_tx'].values, g0['total_tx'].values, alternative='two-sided'
    )
    n1, n0 = len(g1), len(g0)
    r = 1 - 2 * stat / (n1 * n0)   # rank-biserial correlation

    return {
        'group1': group1_label, 'n1': n1, 'pdr1': pdr1,
        'group0': group0_label or f'not {group1_label}', 'n0': n0, 'pdr0': pdr0,
        'diff_pp': diff, 'p_value': p, 'effect_r': r,
        'significant': p < 0.05
    }

def print_comparison(result, title=''):
    sig = '✓ SIGNIFICANT' if result['significant'] else '✗ not significant'
    print(f"  {title}")
    print(f"    {result['group1']:>20}: {result['pdr1']:.2f}%  (n={result['n1']:,})")
    print(f"    {result['group0']:>20}: {result['pdr0']:.2f}%  (n={result['n0']:,})")
    print(f"    Difference        : {result['diff_pp']:+.2f} pp")
    print(f"    p-value           : {result['p_value']:.4f}  {sig}")
    print(f"    Effect size (r)   : {result['effect_r']:.4f}")
    print()

print('Statistical framework ready.')
print('Mann-Whitney U + rank-biserial correlation for all event comparisons.')

Statistical framework ready.
Mann-Whitney U + rank-biserial correlation for all event comparisons.


## 4 · Temporal Event-Conditioned Reliability  *(§7.3 — RQ2)*

**Hypothesis:** Office hours and weekdays should show lower PDR than off-peak periods  
due to higher human activity and increased RF interference from electronic devices.

Temporal events: E2 (weekday), E4 (time-of-day), E5 (office hours), E6 (season)

In [5]:
print('=' * 60)
print('TEMPORAL EVENTS (§7.3)')
print('=' * 60)

temporal_results = {}

# E2: weekday vs weekend
print('\nE2: Weekday vs Weekend')
r = pdr_compare(df_link, 'e2_is_weekday', 1, 0)
print_comparison(r, 'Weekday (1) vs Weekend (0)')
temporal_results['E2_weekday'] = r

# E4: time-of-day bands
print('E4: Time-of-Day PDR')
for band in ['night', 'morning', 'peak', 'evening']:
    g = df_link[df_link['e4_time_of_day'] == band]
    rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
    pdr = rx / (rx + lost) * 100
    print(f'    {band:<10}: {pdr:.2f}%  (n={rx:,})')

# E5: office hours binary
print('\nE5: Office Hours vs Off-Peak')
r = pdr_compare(df_link, 'e5_office_hours', 1, 0)
print_comparison(r, 'Office hours (1) vs off-peak (0)')
temporal_results['E5_office'] = r

# E6: season
print('E6: Season PDR')
for season in ['autumn', 'winter', 'spring']:
    g = df_link[df_link['e6_season'] == season]
    rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
    pdr = rx / (rx + lost) * 100
    print(f'    {season:<10}: {pdr:.2f}%  (n={rx:,})')

TEMPORAL EVENTS (§7.3)

E2: Weekday vs Weekend
  Weekday (1) vs Weekend (0)
                       1: 96.09%  (n=820,283)
                   not 1: 97.03%  (n=336,124)
    Difference        : -0.94 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0024

E4: Time-of-Day PDR
    night     : 96.83%  (n=338,998)
    morning   : 96.72%  (n=145,160)
    peak      : 95.68%  (n=381,259)
    evening   : 96.53%  (n=290,990)

E5: Office Hours vs Off-Peak
  Office hours (1) vs off-peak (0)
                       1: 95.36%  (n=338,049)
                   not 1: 96.78%  (n=818,358)
    Difference        : -1.42 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0048

E6: Season PDR
    autumn    : 96.70%  (n=328,214)
    winter    : 95.99%  (n=407,201)
    spring    : 96.45%  (n=420,992)


## 5 · Occupancy-Conditioned Reliability  *(§7.4 — RQ2)*

**Hypothesis:** Higher CO₂ (more people) → more RF interference → lower PDR.  
CO₂ is the primary occupancy proxy. This is the most important environmental event family.

In [6]:
print('=' * 60)
print('OCCUPANCY EVENTS — CO2 (§7.4)')
print('=' * 60)

occupancy_results = {}

# E1: CO2 tier — three-way comparison
print('\nE1: CO2 Tier PDR')
co2_pdrs = {}
for tier in ['background', 'moderate', 'high']:
    g = df_link[df_link['e1_co2_tier'] == tier]
    rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
    pdr = rx / (rx + lost) * 100
    co2_pdrs[tier] = pdr
    print(f'    {tier:<12}: {pdr:.2f}%  (n={rx:,})')

# key comparison: high vs background
print('\nE1: High vs Background CO2 (key comparison)')
r = pdr_compare(df_link, 'e1_co2_tier', 'high', 'background')
print_comparison(r, 'High CO2 vs Background')
occupancy_results['E1_high_vs_bg'] = r

# moderate vs background
r = pdr_compare(df_link, 'e1_co2_tier', 'moderate', 'background')
print_comparison(r, 'Moderate CO2 vs Background')
occupancy_results['E1_mod_vs_bg'] = r

# E3: CO2 rising
print('E3: CO2 Rising vs Stable')
r = pdr_compare(df_link, 'e3_co2_rising', 1, 0)
print_comparison(r, 'CO2 rising (1) vs stable (0)')
occupancy_results['E3_rising'] = r

OCCUPANCY EVENTS — CO2 (§7.4)

E1: CO2 Tier PDR
    background  : 96.84%  (n=597,487)
    moderate    : 96.30%  (n=416,678)
    high        : 94.57%  (n=142,242)

E1: High vs Background CO2 (key comparison)
  High CO2 vs Background
                    high: 94.57%  (n=142,242)
              background: 96.84%  (n=597,487)
    Difference        : -2.27 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0096

  Moderate CO2 vs Background
                moderate: 96.30%  (n=416,678)
              background: 96.84%  (n=597,487)
    Difference        : -0.54 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0024

E3: CO2 Rising vs Stable
  CO2 rising (1) vs stable (0)
                       1: 33.48%  (n=1,087)
                   not 1: 96.53%  (n=1,155,320)
    Difference        : -63.05 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.3227



## 6 · Air Quality and Atmospheric Events  *(§7.5 — RQ2)*

**PM2.5:** Activity bursts (cleaning, printing, cooking) — does increased  
particulate matter correlate with increased loss?

**Atmospheric:** Pressure, humidity, temperature — seasonal and HVAC effects.

In [12]:
print('=' * 60)
print('AIR QUALITY AND ATMOSPHERIC EVENTS (§7.5)')
print('=' * 60)

atm_results = {}

# E7: PM2.5 spike
print('\nE7: PM2.5 Spike (device-specific 90th percentile)')
r = pdr_compare(df_link, 'e7_pm25_spike', 1, 0)
print_comparison(r, 'PM2.5 spike (1) vs normal (0)')
atm_results['E7_spike'] = r

# E8: PM2.5 absolute tier
print('E8: PM2.5 Absolute Tier')
for tier in ['clean', 'moderate', 'elevated']:
    g = df_link[df_link['e8_pm25_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<10}: {pdr:.2f}%  (n={rx:,})')

# E9: pressure tier
print('\nE9: Pressure Tier PDR')
for tier in ['low', 'medium_low', 'medium_high', 'high']:
    g = df_link[df_link['e9_pressure_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<15}: {pdr:.2f}%  (n={rx:,})')

# E10: pressure drop
print('\nE10: Pressure Drop Event')
r = pdr_compare(df_link, 'e10_pressure_drop', 1, 0)
print_comparison(r, 'Pressure drop (1) vs stable (0)')
atm_results['E10_drop'] = r

# E11: humidity tier
print('E11: Humidity Tier PDR')
for tier in ['dry', 'normal', 'humid', 'very_humid']:
    g = df_link[df_link['e11_humidity_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<12}: {pdr:.2f}%  (n={rx:,})')

# E12: temperature tier
print('\nE12: Temperature Tier PDR')
for tier in ['cold', 'cool', 'warm', 'hot']:
    g = df_link[df_link['e12_temp_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<8}: {pdr:.2f}%  (n={rx:,})')

AIR QUALITY AND ATMOSPHERIC EVENTS (§7.5)

E7: PM2.5 Spike (device-specific 90th percentile)
  PM2.5 spike (1) vs normal (0)
                       1: 94.28%  (n=115,186)
                   not 1: 96.59%  (n=1,041,221)
    Difference        : -2.31 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0153

E8: PM2.5 Absolute Tier
    clean     : 96.65%  (n=799,081)
    moderate  : 95.99%  (n=343,389)
    elevated  : 89.36%  (n=13,937)

E9: Pressure Tier PDR
    low            : 95.99%  (n=289,146)
    medium_low     : 96.09%  (n=289,341)
    medium_high    : 96.56%  (n=289,222)
    high           : 96.80%  (n=288,698)

E10: Pressure Drop Event
  Pressure drop (1) vs stable (0)
                       1: 6.68%  (n=35)
                   not 1: 96.40%  (n=1,156,372)
    Difference        : -89.72 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.7072

E11: Humidity Tier PDR
    dry         : 96.29%  (n=826,351)
    normal      : 96.57%  (n=

## 7 · Spreading Factor and ToA-Conditioned Reliability  *(§7.6 — RQ2)*

**Hypothesis:** Higher SF → longer ToA → wider collision window → lower PDR.  
SF is the strongest non-environmental predictor. This section also presents  
the novel **SF × CO₂ joint analysis** — the interaction that is this thesis's headline finding.

**Joint finding hypothesis:** During high-occupancy periods, the PDR gap between  
low-SF and high-SF transmissions should widen — because longer ToA collides  
disproportionately with the elevated traffic from multiple active users.

In [13]:
print('=' * 60)
print('SF AND TOA EVENTS (§7.6)')
print('=' * 60)

sf_results = {}

# E13: PDR per individual SF value
print('\nE13: PDR by Individual Spreading Factor')
sf_pdrs = {}
for sf in [7, 8, 9, 10]:
    g = df_link[df_link['e13_sf'] == sf]
    rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
    pdr = rx / (rx + lost) * 100
    sf_pdrs[sf] = pdr
    toa = {7:71.9, 8:133.6, 9:246.8, 10:452.6}[sf]
    print(f'    SF{sf} (ToA={toa}ms): {pdr:.2f}%  (n={rx:,})')

print(f'\n    PDR drop SF7→SF10: {sf_pdrs[7]-sf_pdrs[10]:.2f} pp')

# E14: SF tier binary
print('\nE14: SF Tier Comparison')
r = pdr_compare(df_link, 'e14_sf_tier', 'high_sf', 'low_sf')
print_comparison(r, 'High SF (9-10) vs Low SF (7-8)')
sf_results['E14_tier'] = r

# E14 x E1: JOINT ANALYSIS — SF tier x CO2 tier
print('\n' + '=' * 60)
print('NOVEL JOINT ANALYSIS: SF Tier × CO2 Tier  (Table 7.7)')
print('=' * 60)
print(f"{'CO2 Tier':<15} {'Low SF (7-8)':>14} {'High SF (9-10)':>16} {'Gap (pp)':>10}")
print('-' * 58)

joint_results = {}
for co2_tier in ['background', 'moderate', 'high']:
    g_low  = df_link[(df_link['e14_sf_tier']=='low_sf')  & (df_link['e1_co2_tier']==co2_tier)]
    g_high = df_link[(df_link['e14_sf_tier']=='high_sf') & (df_link['e1_co2_tier']==co2_tier)]

    def pdr(g):
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        return rx / (rx + lost) * 100 if (rx+lost) > 0 else 0

    pdr_low  = pdr(g_low)
    pdr_high = pdr(g_high)
    gap      = pdr_low - pdr_high
    joint_results[co2_tier] = {'low_sf': pdr_low, 'high_sf': pdr_high, 'gap': gap}
    print(f"    {co2_tier:<15} {pdr_low:>10.2f}%   {pdr_high:>12.2f}%   {gap:>+8.2f} pp")

# TODO NO E15 ANALYSIS???!!!
print('-' * 58)
gap_bg = joint_results['background']['gap']
gap_hi = joint_results['high']['gap']
print(f'\nInteraction: gap widens from {gap_bg:.2f} pp (background) to {gap_hi:.2f} pp (high CO2)')
print(f'Widening = {gap_hi-gap_bg:.2f} pp — SF×occupancy interaction confirmed.')

SF AND TOA EVENTS (§7.6)

E13: PDR by Individual Spreading Factor
    SF7 (ToA=71.9ms): 98.10%  (n=317,047)
    SF8 (ToA=133.6ms): 97.40%  (n=314,525)
    SF9 (ToA=246.8ms): 97.47%  (n=297,675)
    SF10 (ToA=452.6ms): 91.39%  (n=227,160)

    PDR drop SF7→SF10: 6.71 pp

E14: SF Tier Comparison
  High SF (9-10) vs Low SF (7-8)
                 high_sf: 94.74%  (n=524,835)
                  low_sf: 97.75%  (n=631,572)
    Difference        : -3.01 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0099


NOVEL JOINT ANALYSIS: SF Tier × CO2 Tier  (Table 7.7)
CO2 Tier          Low SF (7-8)   High SF (9-10)   Gap (pp)
----------------------------------------------------------
    background           98.09%          95.38%      +2.71 pp
    moderate             97.58%          94.78%      +2.79 pp
    high                 96.82%          91.97%      +4.86 pp
----------------------------------------------------------

Interaction: gap widens from 2.71 pp (background)

In [ ]:
# TODO NO E16 FOR BURST LOSS ANALYSIS???!!! Or it should be there after logistic regression? Hence we remove it from the events? Since it is a loss type?

## 8 · Signal Context Events  *(§7.5 — RQ2)*

Do packets with weaker signal strength show higher loss rates?  
E17 (RSSI) and E18 (ESP) use the signal quality of the last received packet  
as a proxy for channel state during loss episodes.

In [14]:
print('=' * 60)
print('SIGNAL CONTEXT EVENTS (§7.5)')
print('=' * 60)

signal_results = {}

# E17: RSSI tier
print('\nE17: PDR by RSSI Tier')
for tier in ['weak', 'moderate', 'strong']:
    g = df_link[df_link['e17_rssi_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<10}: {pdr:.2f}%  (n={rx:,})')

r = pdr_compare(df_link, 'e17_rssi_tier', 'weak', 'strong')
print()
print_comparison(r, 'Weak vs Strong RSSI')
signal_results['E17_weak_vs_strong'] = r

# E18: ESP tier
print('E18: PDR by ESP Tier')
for tier in ['low_esp', 'medium_esp', 'high_esp']:
    g = df_link[df_link['e18_esp_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<12}: {pdr:.2f}%  (n={rx:,})')

r = pdr_compare(df_link, 'e18_esp_tier', 'low_esp', 'high_esp')
print()
print_comparison(r, 'Low ESP vs High ESP')
signal_results['E18_low_vs_high'] = r

SIGNAL CONTEXT EVENTS (§7.5)

E17: PDR by RSSI Tier
    weak      : 96.29%  (n=232,755)
    moderate  : 96.50%  (n=336,467)
    strong    : 96.31%  (n=587,185)

  Weak vs Strong RSSI
                    weak: 96.29%  (n=232,755)
                  strong: 96.31%  (n=587,185)
    Difference        : -0.02 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : 0.0048

E18: PDR by ESP Tier
    low_esp     : 96.31%  (n=387,003)
    medium_esp  : 96.57%  (n=387,566)
    high_esp    : 96.19%  (n=381,838)

  Low ESP vs High ESP
                 low_esp: 96.31%  (n=387,003)
                high_esp: 96.19%  (n=381,838)
    Difference        : +0.12 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : 0.0039



## 9 · Complete Event Summary Table  *(Tables 7.x)*

All 18 events ranked by absolute PDR difference.  
This table maps directly to your thesis Chapter 7 summary tables.

In [15]:
print('=' * 60)
print('ALL EVENTS — PDR SUMMARY (ranked by effect size)')
print('=' * 60)

all_results = []

# binary events — compare active vs inactive
binary_events = [
    ('E2',  'e2_is_weekday',   1, 0,            'Weekday vs Weekend'),
    ('E3',  'e3_co2_rising',   1, 0,            'CO2 rising vs stable'),
    ('E5',  'e5_office_hours', 1, 0,            'Office hours vs off-peak'),
    ('E7',  'e7_pm25_spike',   1, 0,            'PM2.5 spike vs normal'),
    ('E10', 'e10_pressure_drop',1, 0,           'Pressure drop vs stable'),
    ('E14', 'e14_sf_tier', 'high_sf','low_sf',  'High SF vs Low SF'),
    ('E15', 'e15_toa_class','long_toa','short_toa','Long ToA vs Short ToA'),
]

for eid, col, g1, g0, label in binary_events:
    r = pdr_compare(df_link, col, g1, g0)
    all_results.append({
        'Event': eid, 'Comparison': label,
        'PDR_active': r['pdr1'], 'PDR_inactive': r['pdr0'],
        'Diff_pp': r['diff_pp'], 'p_value': r['p_value'],
        'Effect_r': r['effect_r'], 'Significant': r['significant']
    })

# categorical events — compare extreme tiers
cat_events = [
    ('E1',  'e1_co2_tier',      'high',      'background', 'High CO2 vs Background'),
    ('E4',  'e4_time_of_day',   'peak',      'night',      'Peak vs Night'),
    ('E6',  'e6_season',        'winter',    'autumn',     'Winter vs Autumn'),
    ('E8',  'e8_pm25_tier',     'elevated',  'clean',      'Elevated vs Clean PM2.5'),
    ('E9',  'e9_pressure_tier', 'high',      'low',        'High vs Low Pressure'),
    ('E11', 'e11_humidity_tier','very_humid','dry',         'Very Humid vs Dry'),
    ('E12', 'e12_temp_tier',    'hot',       'cold',       'Hot vs Cold'),
    ('E13', 'e13_sf',           10,          7,            'SF10 vs SF7'),
    ('E17', 'e17_rssi_tier',    'weak',      'strong',     'Weak vs Strong RSSI'),
    ('E18', 'e18_esp_tier',     'low_esp',   'high_esp',   'Low vs High ESP'),
]

for eid, col, g1, g0, label in cat_events:
    r = pdr_compare(df_link, col, g1, g0)
    all_results.append({
        'Event': eid, 'Comparison': label,
        'PDR_active': r['pdr1'], 'PDR_inactive': r['pdr0'],
        'Diff_pp': r['diff_pp'], 'p_value': r['p_value'],
        'Effect_r': r['effect_r'], 'Significant': r['significant']
    })

results_df = pd.DataFrame(all_results).sort_values('Diff_pp')

print(f"{'Event':<6} {'Comparison':<35} {'PDR_active':>11} {'PDR_inactive':>13} {'Diff(pp)':>10} {'p-value':>9} {'r':>7} {'Sig':>5}")
print('-' * 100)
for _, row in results_df.iterrows():
    sig = '✓' if row['Significant'] else '✗'
    print(f"{row['Event']:<6} {row['Comparison']:<35} {row['PDR_active']:>10.2f}% "
          f"{row['PDR_inactive']:>12.2f}% {row['Diff_pp']:>+9.2f} "
          f"{row['p_value']:>9.4f} {row['Effect_r']:>7.4f} {sig:>5}")

ALL EVENTS — PDR SUMMARY (ranked by effect size)
Event  Comparison                           PDR_active  PDR_inactive   Diff(pp)   p-value       r   Sig
----------------------------------------------------------------------------------------------------
E11    Very Humid vs Dry                         0.00%        96.29%    -96.29       nan     nan     ✗
E10    Pressure drop vs stable                   6.68%        96.40%    -89.72    0.0000 -0.7072     ✓
E3     CO2 rising vs stable                     33.48%        96.53%    -63.05    0.0000 -0.3227     ✓
E8     Elevated vs Clean PM2.5                  89.36%        96.65%     -7.29    0.0000 -0.0268     ✓
E13    SF10 vs SF7                              91.39%        98.10%     -6.71    0.0000 -0.0195     ✓
E14    High SF vs Low SF                        94.74%        97.75%     -3.01    0.0000 -0.0099     ✓
E15    Long ToA vs Short ToA                    94.74%        97.75%     -3.01    0.0000 -0.0099     ✓
E7     PM2.5 spike vs nor

## 10 · Logistic Regression — Loss Risk Model  *(§7.7 — RQ3)*

**Research Question 3:** Which conditions are most strongly associated with loss risk?

The model predicts the binary outcome: did a loss occur in this interval?  
```
y_i = 1[mac_to_radio_loss > 0]
```

Predictors: all 18 events + device-ID fixed effects (ED0 as reference).  
Fitted on the full filtered dataset. Odds ratios computed as OR = exp(β).

**OR > 1** → increased loss probability when event is active  
**OR < 1** → decreased loss probability when event is active

In [1]:
print('=' * 60)
print('LOGISTIC REGRESSION — LOSS RISK MODEL (§7.7)')
print('=' * 60)

# binary outcome: did any loss occur?
df_link = df_link.copy()
df_link['loss_occurred'] = (df_link['mac_to_radio_loss'] > 0).astype(int)

# encode categorical event columns as dummies
feature_df = pd.get_dummies(df_link[[
    'e1_co2_tier', 'e2_is_weekday', 'e3_co2_rising',
    'e4_time_of_day', 'e5_office_hours', 'e6_season',
    'e7_pm25_spike', 'e8_pm25_tier',
    'e9_pressure_tier', 'e10_pressure_drop', 'e11_humidity_tier', 'e12_temp_tier',
    'e13_sf', 'e14_sf_tier', 'e15_toa_class',
    'e17_rssi_tier', 'e18_esp_tier',
    'device_id'
]], drop_first=True).astype(float)

X = feature_df.values
y = df_link['loss_occurred'].values

# fit logistic regression
model = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0)
model.fit(X, y)

# odds ratios
coefs = pd.Series(model.coef_[0], index=feature_df.columns)
odds_ratios = np.exp(coefs).sort_values(ascending=False)

print(f'\nModel fitted on {len(y):,} intervals')
print(f'Loss rate: {y.mean()*100:.2f}%')
print(f'\n{"Predictor":<40} {"Odds Ratio":>12} {"Direction":>12}')
print('-' * 68)
for feat, OR in odds_ratios.items():
    direction = '↑ higher risk' if OR > 1 else '↓ lower risk'
    print(f'{feat:<40} {OR:>12.4f} {direction:>12}')

LOGISTIC REGRESSION — LOSS RISK MODEL (§7.7)


NameError: name 'df_link' is not defined

## 11 · Burst Loss Risk Model  *(§7.7 — RQ3)*

Same logistic regression framework but predicting **burst loss** (large_burst):  
```
y_i = 1[e16_loss_type == 'large_burst']
```

Burst-loss odds ratios are systematically larger than packet-loss odds ratios  
for the same events — confirming that high-occupancy and high-SF conditions  
not only increase loss probability but disproportionately increase burst severity.

In [17]:
print('=' * 60)
print('LOGISTIC REGRESSION — BURST RISK MODEL (§7.7)')
print('=' * 60)

# binary outcome: was this a large burst?
df_link['burst_occurred'] = (df_link['e16_loss_type'] == 'large_burst').astype(int)

y_burst = df_link['burst_occurred'].values

model_burst = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0)
model_burst.fit(X, y_burst)

coefs_burst = pd.Series(model_burst.coef_[0], index=feature_df.columns)
or_burst = np.exp(coefs_burst).sort_values(ascending=False)

print(f'\nModel fitted on {len(y_burst):,} intervals')
print(f'Burst rate: {y_burst.mean()*100:.2f}%')
print(f'\n{"Predictor":<40} {"OR (loss)":>10} {"OR (burst)":>12} {"Burst/Loss ratio":>16}')
print('-' * 82)
for feat in odds_ratios.index:
    or_l = odds_ratios[feat]
    or_b = or_burst[feat]
    ratio = or_b / or_l if or_l != 0 else 0
    print(f'{feat:<40} {or_l:>10.4f} {or_b:>12.4f} {ratio:>16.2f}x')

LOGISTIC REGRESSION — BURST RISK MODEL (§7.7)

Model fitted on 1,156,407 intervals
Burst rate: 0.16%

Predictor                                 OR (loss)   OR (burst) Burst/Loss ratio
----------------------------------------------------------------------------------
e3_co2_rising                               37.5910      68.3115             1.82x
e8_pm25_tier_elevated                        1.9935       2.3351             1.17x
e10_pressure_drop                            1.8958       1.6968             0.90x
e17_rssi_tier_weak                           1.6994       0.6878             0.40x
e7_pm25_spike                                1.5517       1.1695             0.75x
e13_sf                                       1.5448       1.0051             0.65x
e11_humidity_tier_humid                      1.5097       1.1793             0.78x
e1_co2_tier_high                             1.3585       1.3423             0.99x
e6_season_spring                             1.2855       0.7495     

## 12 · Key Findings Summary  *(§7 summary)*

Consolidates all results into thesis-ready statements for Chapter 7.

In [18]:
print('=' * 60)
print('KEY FINDINGS SUMMARY')
print('=' * 60)

print('\n--- RQ1: Campaign-Wide Reliability ---')
print(f'PDR_link (true radio): {pdr_link:.2f}%')
print(f'PDR_raw  (all losses): {pdr_raw:.2f}%')
print(f'Decomposition gap    : {pdr_link-pdr_raw:.2f} pp')
print(f'PDR range across devices: {pdr_range:.2f} pp  → collision-dominated evidence')

print('\n--- RQ2: Event-Conditioned PDR ---')
print('Top findings from event analysis:')
top3 = results_df.head(3)
for _, row in top3.iterrows():
    sig = 'p<0.05' if row['Significant'] else 'n.s.'
    print(f"  {row['Event']}: {row['Comparison']} → {row['Diff_pp']:+.2f} pp  ({sig})")

print(f'\nSF×CO2 interaction:')
print(f'  Background CO2: gap = {joint_results["background"]["gap"]:.2f} pp')
print(f'  High CO2      : gap = {joint_results["high"]["gap"]:.2f} pp')
print(f'  Widening      : {joint_results["high"]["gap"]-joint_results["background"]["gap"]:.2f} pp → interaction confirmed')

print('\n--- RQ3: Strongest Predictors ---')
print('Top 5 loss-risk predictors (by OR):')
for feat, OR in odds_ratios.head(5).items():
    print(f'  {feat:<40}: OR = {OR:.4f}')
print('\nTop 5 burst-risk predictors (by OR):')
for feat, OR in or_burst.head(5).items():
    print(f'  {feat:<40}: OR = {OR:.4f}')

KEY FINDINGS SUMMARY

--- RQ1: Campaign-Wide Reliability ---
PDR_link (true radio): 96.36%
PDR_raw  (all losses): 65.83%
Decomposition gap    : 30.53 pp
PDR range across devices: 0.80 pp  → collision-dominated evidence

--- RQ2: Event-Conditioned PDR ---
Top findings from event analysis:
  E11: Very Humid vs Dry → -96.29 pp  (n.s.)
  E10: Pressure drop vs stable → -89.72 pp  (p<0.05)
  E3: CO2 rising vs stable → -63.05 pp  (p<0.05)

SF×CO2 interaction:
  Background CO2: gap = 2.71 pp
  High CO2      : gap = 4.86 pp
  Widening      : 2.15 pp → interaction confirmed

--- RQ3: Strongest Predictors ---
Top 5 loss-risk predictors (by OR):
  e3_co2_rising                           : OR = 37.5910
  e8_pm25_tier_elevated                   : OR = 1.9935
  e10_pressure_drop                       : OR = 1.8958
  e17_rssi_tier_weak                      : OR = 1.6994
  e7_pm25_spike                           : OR = 1.5517

Top 5 burst-risk predictors (by OR):
  e3_co2_rising                        

## 13 · Save Results

In [ ]:
# save complete results table
results_df.to_csv(DATA_DIR / '3_results.csv', index=False)

# save OR tables
or_table = pd.DataFrame({
    'predictor': odds_ratios.index,
    'OR_loss': odds_ratios.values,
    'OR_burst': [or_burst[f] for f in odds_ratios.index]
})
or_table.to_csv(DATA_DIR / '3_odds_ratios.csv', index=False)

# save joint SF x CO2 results
joint_df = pd.DataFrame(joint_results).T
joint_df.index.name = 'co2_tier'
joint_df.to_csv(DATA_DIR / '3_joint_sf_co2.csv')

print(f'Saved: {DATA_DIR}/3_results.csv')
print(f'Saved: {DATA_DIR}/3_odds_ratios.csv')
print(f'Saved: {DATA_DIR}/3_joint_sf_co2.csv')